# Healthcare Telemetry Data Pipeline

This notebook focuses on loading, exploring, cleaning, and preprocessing the healthcare telemetry dataset before it is used for distributed processing with Apache Spark.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns
pd.set_option("display.max_columns", None)

# Set random seed so our generated values are reproducible
np.random.seed(42)

## 1. Dataset Loading

The healthcare telemetry dataset is loaded into a Pandas DataFrame for initial exploration and preprocessing.

In [3]:
df = pd.read_csv("datasets/healthcare_iot_target_dataset.csv")

## 2. Initial Data Exploration

The dataset is inspected to understand its dimensions, column names, data types, and summary statistics before preprocessing.

In [4]:
df.head()

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
0,BAG_0001,2024-01-11 00:00:00,Hospital_3,B-,RBC,3.500000,3.897717,4.023182,0.098555,0.035499,-0.002283,58.093670,3.140870,-0.014640,-2.558288,0.047550,0.442093,0.994176
1,BAG_0001,2024-01-11 01:00:00,Hospital_3,B-,RBC,3.500000,4.369670,4.446651,0.212670,0.054836,0.005992,60.000000,2.260903,0.062867,-1.911114,0.031802,0.445181,0.987408
2,BAG_0001,2024-01-11 02:00:00,Hospital_3,B-,RBC,4.339104,5.230218,5.548213,0.083685,0.061751,0.008715,60.000000,2.880617,-0.131211,-2.807997,0.042688,0.454198,0.980271
3,BAG_0001,2024-01-11 03:00:00,Hospital_3,B-,RBC,3.500000,4.163157,4.337382,0.120395,0.053772,0.009840,59.834760,2.848335,-0.053497,-1.915956,0.047537,0.457398,0.973170
4,BAG_0001,2024-01-11 04:00:00,Hospital_3,B-,RBC,3.500000,4.146593,4.162168,0.090862,0.051061,0.002331,57.434017,2.831188,-0.097074,-2.945096,0.047115,0.578019,0.966543


In [5]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 288000
Columns: 18


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 288000 entries, 0 to 287999
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   bag_id             288000 non-null  str    
 1   timestamp          288000 non-null  str    
 2   route              288000 non-null  str    
 3   blood_type         288000 non-null  str    
 4   product_type       288000 non-null  str    
 5   temp_mean          286560 non-null  float64
 6   temp_min           286560 non-null  float64
 7   temp_max           286560 non-null  float64
 8   temp_std           286560 non-null  float64
 9   frac_temp_above_6  286560 non-null  float64
 10  frac_temp_above_8  286560 non-null  float64
 11  hum_mean           286560 non-null  float64
 12  hum_std            286560 non-null  float64
 13  door_count         286560 non-null  float64
 14  light_mean_abs     286560 non-null  float64
 15  accel_rms          286560 non-null  float64
 16  handling_stre

In [7]:
df.describe(include="all")

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
count,288000,288000,288000,288000,288000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,286560.000000,2.780670e+05
unique,300,1200,4,8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,BAG_0001,2024-01-11 00:00:00,Hospital_2,O+,RBC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,960,300,76800,44160,288000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,4.216503,2.166390,4.894544,0.164945,-0.149786,0.045858,61.939062,2.752157,0.723925,-4.107267,0.051531,0.925400,7.640571e-02
std,NaN,NaN,NaN,NaN,NaN,1.040617,1.246633,0.777882,0.073410,0.172336,0.050349,6.198165,0.591208,0.703757,5.883978,0.023350,0.274962,1.815445e-01
min,NaN,NaN,NaN,NaN,NaN,1.403712,-0.613156,2.055640,-0.011957,-0.407949,-0.056428,50.000000,0.796968,-0.531173,-14.239460,0.001464,0.392765,2.910909e-09
25%,NaN,NaN,NaN,NaN,NaN,3.500000,1.243144,4.419963,0.109384,-0.296946,0.020716,57.452103,2.326297,0.181869,-9.148206,0.035971,0.729480,9.282546e-05
50%,NaN,NaN,NaN,NaN,NaN,3.614235,1.760677,4.865654,0.163963,-0.228825,0.049914,60.000000,2.758275,0.647714,-5.341520,0.046698,0.885945,1.556327e-03
75%,NaN,NaN,NaN,NaN,NaN,4.905346,3.117253,5.294858,0.213603,0.007293,0.070408,66.465104,3.220252,1.023389,-0.175563,0.060834,1.105993,3.509507e-02


In [8]:
df.columns.tolist()

['bag_id',
 'timestamp',
 'route',
 'blood_type',
 'product_type',
 'temp_mean',
 'temp_min',
 'temp_max',
 'temp_std',
 'frac_temp_above_6',
 'frac_temp_above_8',
 'hum_mean',
 'hum_std',
 'door_count',
 'light_mean_abs',
 'accel_rms',
 'handling_stress',
 'Health_Index']

## 3. Data Quality Assessment

Before cleaning the dataset, its quality is assessed by checking for missing values, duplicate records, and inconsistent data types. These checks help identify issues that need to be addressed during preprocessing and ensure that the dataset is reliable for further analysis.

In [9]:
df.isnull().sum()

bag_id                  0
timestamp               0
route                   0
blood_type              0
product_type            0
temp_mean            1440
temp_min             1440
temp_max             1440
temp_std             1440
frac_temp_above_6    1440
frac_temp_above_8    1440
hum_mean             1440
hum_std              1440
door_count           1440
light_mean_abs       1440
accel_rms            1440
handling_stress      1440
Health_Index         9933
dtype: int64

In [10]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 288000
Number of columns: 18


In [11]:
missing_values = df.isnull().sum()

missing_values

bag_id                  0
timestamp               0
route                   0
blood_type              0
product_type            0
temp_mean            1440
temp_min             1440
temp_max             1440
temp_std             1440
frac_temp_above_6    1440
frac_temp_above_8    1440
hum_mean             1440
hum_std              1440
door_count           1440
light_mean_abs       1440
accel_rms            1440
handling_stress      1440
Health_Index         9933
dtype: int64

In [12]:
missing_values[missing_values > 0]

temp_mean            1440
temp_min             1440
temp_max             1440
temp_std             1440
frac_temp_above_6    1440
frac_temp_above_8    1440
hum_mean             1440
hum_std              1440
door_count           1440
light_mean_abs       1440
accel_rms            1440
handling_stress      1440
Health_Index         9933
dtype: int64

In [13]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

Duplicate rows: 0


In [14]:
df.dtypes

bag_id                   str
timestamp                str
route                    str
blood_type               str
product_type             str
temp_mean            float64
temp_min             float64
temp_max             float64
temp_std             float64
frac_temp_above_6    float64
frac_temp_above_8    float64
hum_mean             float64
hum_std              float64
door_count           float64
light_mean_abs       float64
accel_rms            float64
handling_stress      float64
Health_Index         float64
dtype: object

In [15]:
df[df["temp_mean"].isnull()].head(10)

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
21,BAG_0001,2024-01-11 21:00:00,Hospital_3,B-,RBC,NaN,3.469433,3.770206,0.020027,0.023100,0.015538,58.580627,2.842843,0.108682,-2.877985,0.037172,0.653812,0.749987
299,BAG_0001,2024-01-23 11:00:00,Hospital_3,B-,RBC,NaN,1.062688,4.448363,0.076488,-0.116047,0.043488,57.722997,3.471751,1.706418,4.060012,0.084053,0.695208,0.026833
448,BAG_0001,2024-01-29 16:00:00,Hospital_3,B-,RBC,NaN,0.919956,5.311726,0.205569,-0.281653,0.079670,59.306939,2.525809,0.623399,-11.651969,0.026827,0.922935,0.008425
737,BAG_0001,2024-02-10 17:00:00,Hospital_3,B-,RBC,NaN,1.297787,5.255338,0.214547,-0.292267,0.103246,60.000000,3.405455,0.335856,-7.627880,0.054741,0.910723,0.000543
771,BAG_0001,2024-02-12 03:00:00,Hospital_3,B-,RBC,NaN,1.373637,5.347122,0.218433,-0.284342,0.092808,60.000000,2.321966,0.586920,-10.062632,0.053951,0.888721,0.000395
830,BAG_0001,2024-02-14 14:00:00,Hospital_3,B-,RBC,NaN,1.195131,5.324562,0.206221,-0.279713,0.094379,NaN,3.479932,1.575952,-2.409686,0.060784,0.994035,NaN
1051,BAG_0002,2024-01-04 19:00:00,Hospital_1,A-,RBC,NaN,3.634423,4.355961,0.134426,-0.044439,0.001578,52.979396,1.819050,0.980299,15.597847,0.057712,0.805032,0.220443
1212,BAG_0002,2024-01-11 12:00:00,Hospital_1,A-,RBC,NaN,3.477713,3.613865,0.149718,0.040009,0.020916,54.198767,2.272977,-0.255499,-2.514256,0.026812,0.668185,0.033712
1271,BAG_0002,2024-01-13 23:00:00,Hospital_1,A-,RBC,NaN,2.839742,3.976889,0.114976,0.052024,0.020469,55.660694,2.627039,1.129715,8.275713,0.082430,0.696655,0.013819
1761,BAG_0002,2024-02-03 09:00:00,Hospital_1,A-,RBC,NaN,1.297302,4.245693,0.207312,-0.303677,0.067677,60.000000,3.564258,1.682188,2.482414,0.070877,1.366016,0.000045


## 4. Data Cleaning and Preprocessing

After assessing the quality of the dataset, preprocessing steps are performed to improve data consistency. This includes handling missing values, converting columns to appropriate data types, standardizing column names, and preparing the dataset for distributed processing using Apache Spark.

In [16]:
# Fill missing temp_mean using the average of temp_min and temp_max
df["temp_mean"] = df["temp_mean"].fillna(
    (df["temp_min"] + df["temp_max"]) / 2
)

In [17]:
df["temp_mean"].isnull().sum()

np.int64(12)

In [18]:
df[df["temp_mean"].isnull()]

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
26694,BAG_0028,2024-02-06 06:00:00,Hospital_3,A-,RBC,NaN,1.782808,NaN,0.251187,-0.254836,0.068074,59.815444,2.320862,0.272266,-6.470264,0.016643,0.834979,6.414014e-04
48596,BAG_0051,2024-01-29 20:00:00,Hospital_4,O-,RBC,NaN,NaN,5.094238,0.231203,-0.278684,0.072281,64.703062,3.073268,0.626740,-11.865895,0.049222,1.230823,3.492555e-04
67174,BAG_0070,2024-02-10 22:00:00,Hospital_3,O-,RBC,NaN,NaN,4.755317,0.260971,-0.310314,0.112488,66.547056,3.179295,0.193159,-9.109120,0.054377,1.352911,5.807758e-07
68651,BAG_0072,2024-01-31 11:00:00,Hospital_1,B+,RBC,NaN,NaN,4.141983,0.154396,-0.316977,0.047743,56.916276,2.585641,1.665302,-0.544295,0.076986,1.283199,3.104623e-03
94764,BAG_0099,2024-02-03 12:00:00,Hospital_3,O+,RBC,NaN,1.447896,NaN,0.164466,-0.327126,0.072347,60.000000,3.130794,0.838690,-8.243767,0.031891,0.860513,1.392053e-03
101586,BAG_0106,2024-02-09 18:00:00,Hospital_1,A+,RBC,NaN,1.562926,NaN,0.271875,-0.282268,0.075179,59.456110,2.706063,1.645881,0.935631,0.068933,1.033091,1.559423e-04
170108,BAG_0178,2024-01-16 20:00:00,Hospital_2,O+,RBC,NaN,2.329697,NaN,0.144224,0.006368,0.015995,57.041176,2.817745,-0.094483,-3.248483,0.036142,0.645546,8.372048e-02
193327,BAG_0202,2024-01-23 07:00:00,Hospital_1,B+,RBC,NaN,NaN,5.493049,0.001109,-0.137177,0.044689,63.894280,2.937228,0.826546,-7.470831,0.031477,0.933753,3.838024e-04
207593,BAG_0217,2024-01-13 17:00:00,Hospital_3,A+,RBC,NaN,3.451786,NaN,0.144210,0.078144,0.027771,56.597318,2.441926,1.144046,8.341958,0.060872,0.491373,8.915571e-02
212065,BAG_0221,2024-02-10 01:00:00,Hospital_3,A+,RBC,NaN,0.872398,NaN,0.299755,-0.293773,0.081504,73.645050,2.916459,2.634236,4.934135,0.102727,1.220086,1.527922e-05


In [19]:
df["temp_mean"] = df["temp_mean"].fillna(df["temp_mean"].median())

In [20]:
df["temp_mean"].isnull().sum()

np.int64(0)

In [21]:
sensor_columns = [
    "temp_min", "temp_max", "temp_std",
    "frac_temp_above_6", "frac_temp_above_8",
    "hum_mean", "hum_std",
    "door_count",
    "light_mean_abs",
    "accel_rms",
    "handling_stress"
]

df[sensor_columns].isnull().sum()

temp_min             1440
temp_max             1440
temp_std             1440
frac_temp_above_6    1440
frac_temp_above_8    1440
hum_mean             1440
hum_std              1440
door_count           1440
light_mean_abs       1440
accel_rms            1440
handling_stress      1440
dtype: int64

In [22]:
df[df["temp_min"].isnull()].head()

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
16,BAG_0001,2024-01-11 16:00:00,Hospital_3,B-,RBC,3.500000,NaN,4.456065,0.057048,0.044272,0.015391,56.851765,2.163862,-0.178203,-3.262067,0.028832,0.462198,0.806966
262,BAG_0001,2024-01-21 22:00:00,Hospital_3,B-,RBC,3.844551,NaN,5.128994,0.128565,-0.110319,0.063417,57.652101,3.250834,0.312105,-10.350727,0.027925,0.619632,0.038200
466,BAG_0001,2024-01-30 10:00:00,Hospital_3,B-,RBC,3.500000,NaN,5.403687,0.142532,-0.309402,0.068871,59.559355,2.657673,0.367258,-11.439637,0.032627,0.814386,0.007315
624,BAG_0001,2024-02-06 00:00:00,Hospital_3,B-,RBC,3.500000,NaN,5.640842,0.162396,-0.295843,0.061854,60.000000,3.363186,0.386365,-6.109596,0.022254,0.893927,0.001820
787,BAG_0001,2024-02-12 19:00:00,Hospital_3,B-,RBC,3.500000,NaN,4.555587,0.230314,-0.321991,0.084865,60.000000,2.217605,0.580754,-10.560840,0.062097,0.837353,0.000342


In [23]:
df[df["hum_mean"].isnull()].head()

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
39,BAG_0001,2024-01-12 15:00:00,Hospital_3,B-,RBC,3.500000,3.708874,4.033025,0.055384,0.020763,0.011933,NaN,2.495537,0.177861,-0.142054,0.031987,1.113376,NaN
830,BAG_0001,2024-02-14 14:00:00,Hospital_3,B-,RBC,3.259846,1.195131,5.324562,0.206221,-0.279713,0.094379,NaN,3.479932,1.575952,-2.409686,0.060784,0.994035,NaN
1303,BAG_0002,2024-01-15 07:00:00,Hospital_1,A-,RBC,3.500000,3.329133,NaN,0.110526,0.076943,0.021243,NaN,2.228322,0.598973,7.823085,0.094426,0.645296,NaN
1548,BAG_0002,2024-01-25 12:00:00,Hospital_1,A-,RBC,3.500000,0.967761,4.624834,0.025281,-0.271376,0.048484,NaN,2.240325,2.213691,-6.828174,0.082328,0.877560,NaN
1754,BAG_0002,2024-02-03 02:00:00,Hospital_1,A-,RBC,3.500000,0.740679,4.318072,0.286795,-0.288027,0.058453,NaN,2.751567,0.606475,-8.702160,0.038863,0.890332,NaN


In [24]:
sensor_columns = [
    "temp_min",
    "temp_max",
    "temp_std",
    "frac_temp_above_6",
    "frac_temp_above_8",
    "hum_mean",
    "hum_std",
    "door_count",
    "light_mean_abs",
    "accel_rms",
    "handling_stress"
]

for column in sensor_columns:
    df[column] = df[column].fillna(df[column].median())

In [25]:
df[sensor_columns].isnull().sum()

temp_min             0
temp_max             0
temp_std             0
frac_temp_above_6    0
frac_temp_above_8    0
hum_mean             0
hum_std              0
door_count           0
light_mean_abs       0
accel_rms            0
handling_stress      0
dtype: int64

In [26]:
df["Health_Index"].isnull().sum()

np.int64(9933)

In [27]:
df[df["Health_Index"].isnull()].head(10)

,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,Health_Index
39,BAG_0001,2024-01-12 15:00:00,Hospital_3,B-,RBC,3.500000,3.708874,4.033025,0.055384,0.020763,0.011933,60.000000,2.495537,0.177861,-0.142054,0.031987,1.113376,NaN
47,BAG_0001,2024-01-12 23:00:00,Hospital_3,B-,RBC,3.500000,3.551899,4.323738,0.106801,0.057970,0.024970,57.837902,2.781615,0.647714,-0.417029,0.045646,0.540179,NaN
69,BAG_0001,2024-01-13 21:00:00,Hospital_3,B-,RBC,3.500000,3.375631,4.438882,0.076592,0.063774,0.049914,58.309204,2.798305,1.081304,11.454278,0.104446,0.820589,NaN
175,BAG_0001,2024-01-18 07:00:00,Hospital_3,B-,RBC,3.500000,2.342207,4.609975,0.175012,0.004762,0.007333,59.303537,3.594461,0.647714,5.655985,0.072632,0.669588,NaN
223,BAG_0001,2024-01-20 07:00:00,Hospital_3,B-,RBC,4.004583,1.859954,4.660929,0.124441,-0.016605,0.022338,59.755724,2.677681,1.405026,-5.341520,0.122796,1.116927,NaN
243,BAG_0001,2024-01-21 03:00:00,Hospital_3,B-,RBC,3.505240,1.752657,4.491385,0.164781,-0.083551,0.049914,58.378116,3.165457,0.166912,-9.134278,0.035905,0.664800,NaN
294,BAG_0001,2024-01-23 06:00:00,Hospital_3,B-,RBC,3.500000,1.244390,4.959314,0.084662,-0.228825,0.049587,56.927100,2.248619,0.811824,-6.891284,0.057866,0.761053,NaN
314,BAG_0001,2024-01-24 02:00:00,Hospital_3,B-,RBC,3.500000,1.149825,5.266449,0.038437,-0.155010,0.032268,58.136425,3.371694,0.809470,-9.176987,0.035613,0.885945,NaN
391,BAG_0001,2024-01-27 07:00:00,Hospital_3,B-,RBC,3.500000,1.650052,5.288921,0.113433,-0.315421,0.080147,59.120553,2.071617,0.647714,-12.066405,0.041885,0.784699,NaN
406,BAG_0001,2024-01-27 22:00:00,Hospital_3,B-,RBC,3.500000,1.904368,5.382281,0.115693,-0.283560,0.077706,59.500499,3.538949,0.798672,-5.341520,0.038982,0.985228,NaN


In [28]:
df["Health_Index"].describe()

count    2.780670e+05
mean     7.640571e-02
std      1.815445e-01
min      2.910909e-09
25%      9.282546e-05
50%      1.556327e-03
75%      3.509507e-02
max      9.993902e-01
Name: Health_Index, dtype: float64

In [29]:
df[df["Health_Index"].isnull()]["route"].value_counts()

route
Hospital_2    2670
Hospital_1    2613
Hospital_3    2482
Hospital_4    2168
Name: count, dtype: int64

In [30]:
df[df["Health_Index"].isnull()]["blood_type"].value_counts()

blood_type
O+     1512
A-     1381
A+     1329
AB+    1273
AB-    1234
B+     1172
B-     1110
O-      922
Name: count, dtype: int64

In [31]:
df[df["Health_Index"].isnull()]["product_type"].value_counts()

product_type
RBC    9933
Name: count, dtype: int64

In [32]:
# Fill missing Health_Index values using the median
df["Health_Index"] = df["Health_Index"].fillna(
    df["Health_Index"].median()
)

In [33]:
df.isnull().sum()

bag_id               0
timestamp            0
route                0
blood_type           0
product_type         0
temp_mean            0
temp_min             0
temp_max             0
temp_std             0
frac_temp_above_6    0
frac_temp_above_8    0
hum_mean             0
hum_std              0
door_count           0
light_mean_abs       0
accel_rms            0
handling_stress      0
Health_Index         0
dtype: int64

In [34]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 288000 entries, 0 to 287999
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   bag_id             288000 non-null  str           
 1   timestamp          288000 non-null  datetime64[us]
 2   route              288000 non-null  str           
 3   blood_type         288000 non-null  str           
 4   product_type       288000 non-null  str           
 5   temp_mean          288000 non-null  float64       
 6   temp_min           288000 non-null  float64       
 7   temp_max           288000 non-null  float64       
 8   temp_std           288000 non-null  float64       
 9   frac_temp_above_6  288000 non-null  float64       
 10  frac_temp_above_8  288000 non-null  float64       
 11  hum_mean           288000 non-null  float64       
 12  hum_std            288000 non-null  float64       
 13  door_count         288000 non-null  float64       
 14 

In [36]:
df.columns = df.columns.str.lower()

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 288000 entries, 0 to 287999
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   bag_id             288000 non-null  str           
 1   timestamp          288000 non-null  datetime64[us]
 2   route              288000 non-null  str           
 3   blood_type         288000 non-null  str           
 4   product_type       288000 non-null  str           
 5   temp_mean          288000 non-null  float64       
 6   temp_min           288000 non-null  float64       
 7   temp_max           288000 non-null  float64       
 8   temp_std           288000 non-null  float64       
 9   frac_temp_above_6  288000 non-null  float64       
 10  frac_temp_above_8  288000 non-null  float64       
 11  hum_mean           288000 non-null  float64       
 12  hum_std            288000 non-null  float64       
 13  door_count         288000 non-null  float64       
 14 

## 5. Final Processed Dataset

The cleaned dataset is verified and prepared for distributed processing using Apache Spark.

In [44]:
# Save the cleaned dataset
df.to_csv(
    "datasets/blood_cold_chain_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [45]:
cleaned_df = pd.read_csv("datasets/blood_cold_chain_cleaned.csv")

print("Rows:", cleaned_df.shape[0])
print("Columns:", cleaned_df.shape[1])

cleaned_df.head()

Rows: 288000
Columns: 18


,bag_id,timestamp,route,blood_type,product_type,temp_mean,temp_min,temp_max,temp_std,frac_temp_above_6,frac_temp_above_8,hum_mean,hum_std,door_count,light_mean_abs,accel_rms,handling_stress,health_index
0,BAG_0001,2024-01-11 00:00:00,Hospital_3,B-,RBC,3.500000,3.897717,4.023182,0.098555,0.035499,-0.002283,58.093670,3.140870,-0.014640,-2.558288,0.047550,0.442093,0.994176
1,BAG_0001,2024-01-11 01:00:00,Hospital_3,B-,RBC,3.500000,4.369670,4.446651,0.212670,0.054836,0.005992,60.000000,2.260903,0.062867,-1.911114,0.031802,0.445181,0.987408
2,BAG_0001,2024-01-11 02:00:00,Hospital_3,B-,RBC,4.339104,5.230218,5.548213,0.083685,0.061751,0.008715,60.000000,2.880617,-0.131211,-2.807997,0.042688,0.454198,0.980271
3,BAG_0001,2024-01-11 03:00:00,Hospital_3,B-,RBC,3.500000,4.163157,4.337382,0.120395,0.053772,0.009840,59.834760,2.848335,-0.053497,-1.915956,0.047537,0.457398,0.973170
4,BAG_0001,2024-01-11 04:00:00,Hospital_3,B-,RBC,3.500000,4.146593,4.162168,0.090862,0.051061,0.002331,57.434017,2.831188,-0.097074,-2.945096,0.047115,0.578019,0.966543
